In [42]:
# %% [markdown]
# # SignGRU v3 — Personal Fine-Tuning
#
# **Goal:** Adapt the trained model to your specific hands and lighting.
#
# **Strategy:** Freeze GRU layers → train classifier head only (20 epochs fast),
#               then optionally unfreeze everything for 10 more epochs at low LR.
#
# **Input:**  `my_captures.json`  (exported from the RecordMode UI)
# **Output:** `model.onnx`        (drop-in replacement for public/models/)
#
# | Step | What happens |
# |------|-------------|
# | 1 | Load your captures + existing model weights |
# | 2 | Freeze GRU, train head 20 epochs (~2 min CPU) |
# | 3 | Unfreeze all, fine-tune 10 epochs at LR=1e-4 |
# | 4 | Evaluate — per-class F1 on a personal holdout |
# | 5 | Export new model.onnx |


In [43]:
# ── 0. Install / imports ──────────────────────────────────────────────────────
%pip install torch scikit-learn onnx onnxruntime numpy pandas -q

import json, math, os, random, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

print("Imports OK")


Note: you may need to restart the kernel to use updated packages.
Imports OK



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
# ── 1. Config ─────────────────────────────────────────────────────────────────

CAPTURES_PATH  = "my_captures2.json"          # exported from RecordMode UI
EXISTING_ONNX  = "../../frontend/public/models/model.onnx"      # path to current model
META_PATH      = "../../frontend/public/models/model_meta.json" # sign_names, seq_length, etc.
OUTPUT_ONNX    = "../../frontend/public/models/model.onnx"      # overwrites in place (keep a backup!)
BACKUP_ONNX    = "../../frontend/public/models/model_backup.onnx"

SEQ_LENGTH  = 16
N_FEATURES  = 63
BATCH_SIZE  = 16
HEAD_EPOCHS = 20    # Phase 1: frozen GRU, train head only
FULL_EPOCHS = 10    # Phase 2: unfreeze everything
HEAD_LR     = 3e-3
FULL_LR     = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Load metadata ─────────────────────────────────────────────────────────────
with open(META_PATH) as f:
    meta = json.load(f)

sign_names  = meta["sign_names"]
num_classes = len(sign_names)
label2idx   = {name: i for i, name in enumerate(sign_names)}

print(f"Vocabulary: {num_classes} signs")


Device: cpu
Vocabulary: 129 signs


In [45]:
# ── 2. Load personal captures ─────────────────────────────────────────────────
with open(CAPTURES_PATH) as f:
    raw = json.load(f)

X_personal, y_personal = [], []
skipped = []

for entry in raw["captures"]:
    label = entry["label"]
    if label not in label2idx:
        skipped.append(label)
        continue
    idx = label2idx[label]
    for seq_flat in entry["sequences"]:
        arr = np.array(seq_flat, dtype=np.float32).reshape(SEQ_LENGTH, N_FEATURES)
        X_personal.append(arr)
        y_personal.append(idx)

if skipped:
    print(f"⚠️  Skipped unknown labels: {skipped}")

X_personal = np.array(X_personal, dtype=np.float32)
y_personal = np.array(y_personal, dtype=np.int64)

class_counts = Counter(y_personal.tolist())
print(f"Personal data: {len(X_personal)} sequences across {len(class_counts)} signs")
print("Samples per sign:")
for idx, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {sign_names[idx]:20s}  {count}")

# ── Train / holdout split ─────────────────────────────────────────────────────
# Use 20% of personal data as a holdout to measure improvement
X_tr, X_val, y_tr, y_val = train_test_split(
    X_personal, y_personal,
    test_size=0.2, random_state=42,
    stratify=y_personal if len(class_counts) > 1 else None
)
print(f"\nTrain: {len(X_tr)} | Holdout: {len(X_val)}")


Personal data: 45 sequences across 1 signs
Samples per sign:
  A                     45

Train: 36 | Holdout: 9


In [46]:
# ── 3. Dataset ────────────────────────────────────────────────────────────────

class LandmarkDataset(Dataset):
    def __init__(self, X, y, augment=True):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.augment = augment

    def __len__(self): return len(self.y)

    def __getitem__(self, idx):
        seq = self.X[idx].clone().numpy().reshape(SEQ_LENGTH, 21, 3)
        if self.augment:
            # rotation ±15 degrees (smaller than training — don't overfit)
            if random.random() < 0.6:
                a = random.uniform(-15, 15) * math.pi / 180
                c, s = math.cos(a), math.sin(a)
                R = np.array([[c, -s], [s, c]])
                seq[:, :, :2] = seq[:, :, :2] @ R.T
            # scale jitter
            if random.random() < 0.6:
                seq *= random.uniform(0.88, 1.12)
            # horizontal flip
            if random.random() < 0.5:
                seq[:, :, 0] *= -1
            # gaussian noise
            seq += np.random.normal(0, 0.01, seq.shape).astype(np.float32)
        return torch.FloatTensor(seq.reshape(SEQ_LENGTH, 63)), self.y[idx]

# Weighted sampler so rare classes aren't ignored
cc_tr = Counter(y_tr.tolist())
sample_weights = [1.0 / cc_tr[int(l)] for l in y_tr]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(LandmarkDataset(X_tr, y_tr, augment=True),
                          batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(LandmarkDataset(X_val, y_val, augment=False),
                          batch_size=64, shuffle=False)

print("Datasets OK")


Datasets OK


In [47]:
# ── 4. Model definition (must match v3 exactly) ───────────────────────────────

class SignGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_size)
        self.gru = nn.GRU(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        d = hidden_size * 2
        self.attention = nn.Linear(d, 1)
        self.dropout   = nn.Dropout(dropout)
        self.fc1 = nn.Linear(d * 3, hidden_size)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        self.act = nn.GELU()

    def forward(self, x):
        x   = self.input_norm(x)
        out, _ = self.gru(x)
        attn   = torch.softmax(self.attention(out), dim=1)
        a_pool = (out * attn).sum(dim=1)
        m_pool, _ = out.max(dim=1)
        last   = out[:, -1, :]
        h = torch.cat([a_pool, m_pool, last], dim=1)
        h = self.act(self.fc1(self.dropout(h)))
        return self.fc2(self.dropout(h))

# ── Load original weights from ONNX via onnxruntime ──────────────────────────
# We can't load directly from .onnx into PyTorch, so we need the original
# .pt checkpoint. If you have best_state.pt, use that. Otherwise, re-run
# the original training notebook to produce one, then come back here.

CHECKPOINT_PATH = "../../frontend/public/models/best_state.pt"   # saved PyTorch state dict

HIDDEN_SIZE = meta.get("hidden_size", 192)
NUM_LAYERS  = meta.get("num_layers", 2)

model = SignGRU(N_FEATURES, HIDDEN_SIZE, num_classes, NUM_LAYERS).to(device)

if os.path.exists(CHECKPOINT_PATH):
    state = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(state)
    print(f"✅ Loaded weights from {CHECKPOINT_PATH}")
else:
    print("⚠️  No checkpoint found at", CHECKPOINT_PATH)
    print("   Run the original training notebook and save:")
    print("   torch.save(best_state, '../../frontend/public/models/best_state.pt')")
    print("   Proceeding with random weights (not recommended).")


⚠️  No checkpoint found at ../../frontend/public/models/best_state.pt
   Run the original training notebook and save:
   torch.save(best_state, '../../frontend/public/models/best_state.pt')
   Proceeding with random weights (not recommended).


In [48]:
# ── 5. BASELINE — evaluate before fine-tuning ────────────────────────────────
# This is your before score. Fine-tuning should improve it on personal data.

def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for bx, by in loader:
            _, pred = torch.max(model(bx.to(device)), 1)
            preds.extend(pred.cpu().numpy())
            trues.extend(by.numpy())
    return accuracy_score(trues, preds), preds, trues

baseline_acc, baseline_preds, baseline_trues = evaluate(model, val_loader)
print(f"Baseline accuracy on your personal holdout: {baseline_acc:.4f}")
print("(This is how well the original model knows your hands before fine-tuning)")


Baseline accuracy on your personal holdout: 0.2222
(This is how well the original model knows your hands before fine-tuning)


In [49]:
# ── 6. PHASE 1 — Freeze GRU, train head only ─────────────────────────────────
# Freezing the GRU layers preserves the temporal motion knowledge learned
# from thousands of WLASL videos. Only the classifier head adapts to your
# personal hand geometry and lighting.

print("\n── Phase 1: Frozen GRU ──")

# Freeze everything except fc1, fc2, input_norm
for name, param in model.named_parameters():
    if any(name.startswith(p) for p in ["fc1", "fc2", "attention"]):
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=HEAD_LR, weight_decay=1e-3
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, HEAD_EPOCHS)

best_acc   = baseline_acc
best_state = copy.deepcopy(model.state_dict())

for epoch in range(HEAD_EPOCHS):
    model.train()
    total_loss = 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    if (epoch + 1) % 5 == 0:
        acc, _, _ = evaluate(model, val_loader)
        if acc > best_acc:
            best_acc   = acc
            best_state = copy.deepcopy(model.state_dict())
        print(f"  Ep {epoch+1:2d}/{HEAD_EPOCHS} | Loss {total_loss/len(train_loader):.4f} "
              f"| Val {acc:.4f} | Best {best_acc:.4f}")

print(f"\nPhase 1 done. Best: {best_acc:.4f} (was {baseline_acc:.4f})")



── Phase 1: Frozen GRU ──
Trainable params: 246,658 / 1,208,704 (20.4%)
  Ep  5/20 | Loss 0.5389 | Val 1.0000 | Best 1.0000
  Ep 10/20 | Loss 0.4678 | Val 1.0000 | Best 1.0000
  Ep 15/20 | Loss 0.4608 | Val 1.0000 | Best 1.0000
  Ep 20/20 | Loss 0.4613 | Val 1.0000 | Best 1.0000

Phase 1 done. Best: 1.0000 (was 0.2222)


In [50]:
# ── 7. PHASE 2 — Unfreeze all, fine-tune at low LR ───────────────────────────
# With a good head already, we can safely unfreeze the full model.
# Use a very small LR to nudge the GRU weights without destroying them.

print("\n── Phase 2: Full model unfreeze ──")
model.load_state_dict(best_state)

for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,} / {total:,} (100%)")

optimizer2 = optim.AdamW(model.parameters(), lr=FULL_LR, weight_decay=1e-4)
scheduler2 = optim.lr_scheduler.CosineAnnealingLR(optimizer2, FULL_EPOCHS)

for epoch in range(FULL_EPOCHS):
    model.train()
    total_loss = 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer2.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 0.5)  # tighter clip
        optimizer2.step()
        total_loss += loss.item()
    scheduler2.step()

    acc, _, _ = evaluate(model, val_loader)
    if acc > best_acc:
        best_acc   = acc
        best_state = copy.deepcopy(model.state_dict())
    print(f"  Ep {epoch+1:2d}/{FULL_EPOCHS} | Loss {total_loss/len(train_loader):.4f} "
          f"| Val {acc:.4f} | Best {best_acc:.4f}")

print(f"\nPhase 2 done. Best: {best_acc:.4f}")



── Phase 2: Full model unfreeze ──
Trainable params: 1,208,704 / 1,208,704 (100%)
  Ep  1/10 | Loss 0.5007 | Val 1.0000 | Best 1.0000
  Ep  2/10 | Loss 0.4875 | Val 1.0000 | Best 1.0000
  Ep  3/10 | Loss 0.4656 | Val 1.0000 | Best 1.0000
  Ep  4/10 | Loss 0.4746 | Val 1.0000 | Best 1.0000
  Ep  5/10 | Loss 0.4731 | Val 1.0000 | Best 1.0000
  Ep  6/10 | Loss 0.4730 | Val 1.0000 | Best 1.0000
  Ep  7/10 | Loss 0.4706 | Val 1.0000 | Best 1.0000
  Ep  8/10 | Loss 0.4657 | Val 1.0000 | Best 1.0000
  Ep  9/10 | Loss 0.4595 | Val 1.0000 | Best 1.0000
  Ep 10/10 | Loss 0.4743 | Val 1.0000 | Best 1.0000

Phase 2 done. Best: 1.0000


In [51]:
# ── 8. Final evaluation ───────────────────────────────────────────────────────
model.load_state_dict(best_state)
final_acc, final_preds, final_trues = evaluate(model, val_loader)

present = sorted(set(final_trues))
print(f"\n{'─'*50}")
print(f"Baseline accuracy : {baseline_acc:.4f}")
print(f"Fine-tuned accuracy: {final_acc:.4f}  (+{final_acc - baseline_acc:+.4f})")
print(f"{'─'*50}")
print("\nPer-class report on YOUR personal holdout:")
print(classification_report(
    final_trues, final_preds,
    labels=present,
    target_names=[sign_names[i] for i in present],
    zero_division=0
))



──────────────────────────────────────────────────
Baseline accuracy : 0.2222
Fine-tuned accuracy: 1.0000  (++0.7778)
──────────────────────────────────────────────────

Per-class report on YOUR personal holdout:
              precision    recall  f1-score   support

           A       1.00      1.00      1.00         9

    accuracy                           1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9



In [52]:
# ── 9. Save checkpoint ────────────────────────────────────────────────────────
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
torch.save(best_state, CHECKPOINT_PATH.replace("best_state", "best_state_finetuned"))
print(f"PyTorch checkpoint saved.")


PyTorch checkpoint saved.


In [53]:
# ── 10. Export to ONNX (single-file, no .onnx.data) ─────────────────────────
import onnx, onnxruntime as ort
from onnx.external_data_helper import load_external_data_for_model

# Back up the original model first
if os.path.exists(EXISTING_ONNX):
    import shutil
    shutil.copy(EXISTING_ONNX, BACKUP_ONNX)
    print(f"Backed up original → {BACKUP_ONNX}")

model_cpu = model.cpu().eval()
dummy     = torch.randn(1, SEQ_LENGTH, N_FEATURES)

torch.onnx.export(
    model_cpu, dummy, OUTPUT_ONNX,
    export_params=True, opset_version=18, do_constant_folding=True,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

model_proto = onnx.load(OUTPUT_ONNX)
load_external_data_for_model(model_proto, os.path.dirname(OUTPUT_ONNX))
onnx.save(model_proto, OUTPUT_ONNX, save_as_external_data=False)

data_file = OUTPUT_ONNX + '.data'
if os.path.exists(data_file):
    os.remove(data_file)
    print('Removed stale .data sidecar')

# Verify
sess = ort.InferenceSession(OUTPUT_ONNX)
out  = sess.run(None, {'input': dummy.numpy()})
size_mb = os.path.getsize(OUTPUT_ONNX) / 1024 / 1024
print(f'Single-file ONNX: {OUTPUT_ONNX}  ({size_mb:.2f} MB)')
print(f'Output shape: {out[0].shape}  OK')

# Update metadata
meta['best_val_acc']     = round(float(final_acc), 4)
meta['baseline_acc']     = round(float(baseline_acc), 4)
meta['finetuned']        = True
meta['personal_samples'] = int(len(X_personal))

with open(META_PATH, 'w') as f:
    json.dump(meta, f, indent=4)
print('model_meta.json updated.')


Backed up original → ../../frontend/public/models/model_backup.onnx


C:\Users\harsh\AppData\Local\Temp\ipykernel_27356\227609946.py:14: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `SignGRU([...]` with `torch.export.export(..., strict=False)`...


c:\Users\harsh\AppData\Local\Programs\Python\Python311\Lib\contextlib.py:144: UserWarning: The tensor attributes self.gru._flat_weights[0], self.gru._flat_weights[1], self.gru._flat_weights[2], self.gru._flat_weights[3], self.gru._flat_weights[4], self.gru._flat_weights[5], self.gru._flat_weights[6], self.gru._flat_weights[7], self.gru._flat_weights[8], self.gru._flat_weights[9], self.gru._flat_weights[10], self.gru._flat_weights[11], self.gru._flat_weights[12], self.gru._flat_weights[13], self.gru._flat_weights[14], self.gru._flat_weights[15] were assigned during export. Such attributes must be registered as buffers using the `register_buffer` API (https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer).
  next(self.gen)


[torch.onnx] Obtain model graph for `SignGRU([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


c:\Users\harsh\AppData\Local\Programs\Python\Python311\Lib\copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 1 of general pattern rewrite rules.
Removed stale .data sidecar
Single-file ONNX: ../../frontend/public/models/model.onnx  (4.71 MB)
Output shape: (1, 129)  OK
model_meta.json updated.


Done!
#
Copy the updated `model.onnx` and `model_meta.json` into your frontend's
`public/models/` directory and refresh the app — no other changes needed.
#
If accuracy didn't improve:
- Check that your captures have actual hand movement (not just static noise)
- Record more takes — aim for 20+ per sign
- Vary your distance and angle between takes
- Make sure your normalization.ts matches normalize_landmarks() in this notebook
#
If the model gets worse on signs you didn't record:
- This is catastrophic forgetting — increase `FULL_LR` to `5e-5` and reduce
`FULL_EPOCHS` to 5, or skip Phase 2 entirely and stick with Phase 1 only.
